In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [4]:
df = pd.read_csv("../data/cleaned_data.csv")

In [5]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True)

In [6]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Date range:", df["date"].min(), "to", df["date"].max())

Rows: 27750
Columns: 10
Date range: 2024-05-18 00:00:00 to 2025-11-27 00:00:00


In [7]:
song_lifecycle = (
    df.groupby(["song", "artist"])
      .agg(
          entry_date=("date", "min"),
          exit_date=("date", "max")
      )
      .reset_index()
)

song_lifecycle.head(10)

,song,artist,entry_date,exit_date
0,0 CONFIANZA,Rvfv,2025-06-02,2025-06-23
1,"1 de Enero, Puntacana",Rels B,2025-04-28,2025-04-28
2,1000COSAS,Lola Indigo & Manuel Turizo,2025-03-10,2025-11-13
3,100xCiento,Foreign Teck & Eladio Carrion & De La Ghetto,2025-04-30,2025-05-15
4,14 FEBREROS,Quevedo,2025-11-24,2025-11-27
5,2AM,Sebastian Yatra & Bad Gyal,2025-11-24,2025-11-27
6,2xLL,JC Reyes & YOVNGCHIMI & Slayter,2025-07-12,2025-08-09
7,3 CAÍDAS,SAIKO,2025-04-28,2025-05-05
8,4 PREZ,Cruz Cafuné,2024-05-28,2024-05-29
9,4TO 23,Aitana,2025-07-07,2025-07-16


In [8]:
print("Unique song-artist combinations:",
      len(song_lifecycle))

Unique song-artist combinations: 583


In [9]:
print(
    "Entry date:",
    song_lifecycle["entry_date"].min(),
    "to",
    song_lifecycle["entry_date"].max()
)

print(
    "Exit date:",
    song_lifecycle["exit_date"].min(),
    "to",
    song_lifecycle["exit_date"].max()
)

Entry date: 2024-05-18 00:00:00 to 2025-11-27 00:00:00
Exit date: 2024-05-20 00:00:00 to 2025-11-27 00:00:00


In [10]:
song_lifecycle["days_on_playlist"] = (
    song_lifecycle["exit_date"]
    - song_lifecycle["entry_date"]
).dt.days + 1

song_lifecycle.head(10)

,song,artist,entry_date,exit_date,days_on_playlist
0,0 CONFIANZA,Rvfv,2025-06-02,2025-06-23,22
1,"1 de Enero, Puntacana",Rels B,2025-04-28,2025-04-28,1
2,1000COSAS,Lola Indigo & Manuel Turizo,2025-03-10,2025-11-13,249
3,100xCiento,Foreign Teck & Eladio Carrion & De La Ghetto,2025-04-30,2025-05-15,16
4,14 FEBREROS,Quevedo,2025-11-24,2025-11-27,4
5,2AM,Sebastian Yatra & Bad Gyal,2025-11-24,2025-11-27,4
6,2xLL,JC Reyes & YOVNGCHIMI & Slayter,2025-07-12,2025-08-09,29
7,3 CAÍDAS,SAIKO,2025-04-28,2025-05-05,8
8,4 PREZ,Cruz Cafuné,2024-05-28,2024-05-29,2
9,4TO 23,Aitana,2025-07-07,2025-07-16,10


In [11]:
print(song_lifecycle["days_on_playlist"].describe())

count    583.000000
mean      56.421955
std       88.303991
min        1.000000
25%        4.000000
50%       15.000000
75%       75.500000
max      551.000000
Name: days_on_playlist, dtype: float64


In [12]:
print(
    "Shortest lifecycle:",
    song_lifecycle["days_on_playlist"].min(),
    "days"
)

print(
    "Longest lifecycle:",
    song_lifecycle["days_on_playlist"].max(),
    "days"
)

print(
    "Average lifecycle:",
    round(song_lifecycle["days_on_playlist"].mean(), 2),
    "days"
)

Shortest lifecycle: 1 days
Longest lifecycle: 551 days
Average lifecycle: 56.42 days


In [13]:
peak_position = (
    df.groupby(["song", "artist"])["position"]
      .min()
      .reset_index(name="peak_position")
)

peak_position.head(10)

,song,artist,peak_position
0,0 CONFIANZA,Rvfv,23
1,"1 de Enero, Puntacana",Rels B,45
2,1000COSAS,Lola Indigo & Manuel Turizo,3
3,100xCiento,Foreign Teck & Eladio Carrion & De La Ghetto,13
4,14 FEBREROS,Quevedo,9
5,2AM,Sebastian Yatra & Bad Gyal,31
6,2xLL,JC Reyes & YOVNGCHIMI & Slayter,35
7,3 CAÍDAS,SAIKO,13
8,4 PREZ,Cruz Cafuné,35
9,4TO 23,Aitana,17


In [14]:
song_lifecycle = song_lifecycle.merge(
    peak_position,
    on=["song", "artist"],
    how="left"
)

In [15]:
song_lifecycle[
    ["song", "artist", "entry_date", "exit_date",
     "days_on_playlist", "peak_position"]
].head(10)

,song,artist,entry_date,exit_date,days_on_playlist,peak_position
0,0 CONFIANZA,Rvfv,2025-06-02,2025-06-23,22,23
1,"1 de Enero, Puntacana",Rels B,2025-04-28,2025-04-28,1,45
2,1000COSAS,Lola Indigo & Manuel Turizo,2025-03-10,2025-11-13,249,3
3,100xCiento,Foreign Teck & Eladio Carrion & De La Ghetto,2025-04-30,2025-05-15,16,13
4,14 FEBREROS,Quevedo,2025-11-24,2025-11-27,4,9
5,2AM,Sebastian Yatra & Bad Gyal,2025-11-24,2025-11-27,4,31
6,2xLL,JC Reyes & YOVNGCHIMI & Slayter,2025-07-12,2025-08-09,29,35
7,3 CAÍDAS,SAIKO,2025-04-28,2025-05-05,8,13
8,4 PREZ,Cruz Cafuné,2024-05-28,2024-05-29,2,35
9,4TO 23,Aitana,2025-07-07,2025-07-16,10,17


In [16]:
print("Missing peak positions:",
      song_lifecycle["peak_position"].isnull().sum())

print("Best peak position:",
      song_lifecycle["peak_position"].min())

print("Worst peak position:",
      song_lifecycle["peak_position"].max())

Missing peak positions: 0
Best peak position: 1
Worst peak position: 50


In [17]:
peak_dates = (
    df.loc[
        df.groupby(["song", "artist"])["position"].idxmin(),
        ["song", "artist", "date", "position"]
    ]
    .rename(columns={
        "date": "peak_date"
    })
)

peak_dates.head(10)

,song,artist,peak_date,position
19122,0 CONFIANZA,Rvfv,2025-06-07,23
17144,"1 de Enero, Puntacana",Rels B,2025-04-28,45
14752,1000COSAS,Lola Indigo & Manuel Turizo,2025-03-10,3
17762,100xCiento,Foreign Teck & Eladio Carrion & De La Ghetto,2025-05-11,13
27508,14 FEBREROS,Quevedo,2025-11-24,9
27680,2AM,Sebastian Yatra & Bad Gyal,2025-11-27,31
21384,2xLL,JC Reyes & YOVNGCHIMI & Slayter,2025-07-23,35
17112,3 CAÍDAS,SAIKO,2025-04-28,13
534,4 PREZ,Cruz Cafuné,2024-05-28,35
20616,4TO 23,Aitana,2025-07-07,17


In [18]:
song_lifecycle = song_lifecycle.merge(
    peak_dates[["song", "artist", "peak_date"]],
    on=["song", "artist"],
    how="left"
)

In [19]:
song_lifecycle["time_to_peak_days"] = (
    song_lifecycle["peak_date"]
    - song_lifecycle["entry_date"]
).dt.days

In [20]:
song_lifecycle[
    ["song", "artist", "entry_date",
     "peak_date", "peak_position",
     "time_to_peak_days"]
].head(10)

,song,artist,entry_date,peak_date,peak_position,time_to_peak_days
0,0 CONFIANZA,Rvfv,2025-06-02,2025-06-07,23,5
1,"1 de Enero, Puntacana",Rels B,2025-04-28,2025-04-28,45,0
2,1000COSAS,Lola Indigo & Manuel Turizo,2025-03-10,2025-03-10,3,0
3,100xCiento,Foreign Teck & Eladio Carrion & De La Ghetto,2025-04-30,2025-05-11,13,11
4,14 FEBREROS,Quevedo,2025-11-24,2025-11-24,9,0
5,2AM,Sebastian Yatra & Bad Gyal,2025-11-24,2025-11-27,31,3
6,2xLL,JC Reyes & YOVNGCHIMI & Slayter,2025-07-12,2025-07-23,35,11
7,3 CAÍDAS,SAIKO,2025-04-28,2025-04-28,13,0
8,4 PREZ,Cruz Cafuné,2024-05-28,2024-05-28,35,0
9,4TO 23,Aitana,2025-07-07,2025-07-07,17,0


In [21]:
print("Missing peak dates:",
      song_lifecycle["peak_date"].isnull().sum())

print("Minimum time to peak:",
      song_lifecycle["time_to_peak_days"].min(),
      "days")

print("Maximum time to peak:",
      song_lifecycle["time_to_peak_days"].max(),
      "days")

print("Average time to peak:",
      round(
          song_lifecycle["time_to_peak_days"].mean(),
          2
      ),
      "days")

Missing peak dates: 0
Minimum time to peak: 0 days
Maximum time to peak: 426 days
Average time to peak: 9.51 days


In [22]:
observed_days = (
    df.groupby(["song", "artist"])["date"]
      .nunique()
      .reset_index(name="observed_playlist_days")
)

observed_days.head(10)

,song,artist,observed_playlist_days
0,0 CONFIANZA,Rvfv,20
1,"1 de Enero, Puntacana",Rels B,1
2,1000COSAS,Lola Indigo & Manuel Turizo,245
3,100xCiento,Foreign Teck & Eladio Carrion & De La Ghetto,11
4,14 FEBREROS,Quevedo,4
5,2AM,Sebastian Yatra & Bad Gyal,4
6,2xLL,JC Reyes & YOVNGCHIMI & Slayter,26
7,3 CAÍDAS,SAIKO,8
8,4 PREZ,Cruz Cafuné,2
9,4TO 23,Aitana,9


In [23]:
song_lifecycle = song_lifecycle.merge(
    observed_days,
    on=["song", "artist"],
    how="left"
)

In [24]:
print(
    "Missing observed days:",
    song_lifecycle["observed_playlist_days"].isnull().sum()
)

print(
    "Average observed playlist days:",
    round(
        song_lifecycle["observed_playlist_days"].mean(),
        2
    )
)

print(
    "Maximum observed playlist days:",
    song_lifecycle["observed_playlist_days"].max()
)

Missing observed days: 0
Average observed playlist days: 47.6
Maximum observed playlist days: 538


In [25]:
print(
    "Songs where lifecycle span != observed days:",
    (
        song_lifecycle["days_on_playlist"]
        != song_lifecycle["observed_playlist_days"]
    ).sum()
)

Songs where lifecycle span != observed days: 242


In [26]:
entry_position = (
    df.sort_values("date")
      .groupby(["song", "artist"])
      .first()
      .reset_index()[["song", "artist", "position"]]
      .rename(columns={"position": "entry_position"})
)

song_lifecycle = song_lifecycle.merge(
    entry_position,
    on=["song", "artist"],
    how="left"
)

In [27]:
song_lifecycle["rank_improvement"] = (
    song_lifecycle["entry_position"]
    - song_lifecycle["peak_position"]
)

In [28]:
print(
    song_lifecycle[
        [
            "song",
            "entry_position",
            "peak_position",
            "rank_improvement"
        ]
    ].head(10)
)

                    song  entry_position  peak_position  rank_improvement
0            0 CONFIANZA              40             23                17
1  1 de Enero, Puntacana              45             45                 0
2              1000COSAS               3              3                 0
3             100xCiento              42             13                29
4            14 FEBREROS               9              9                 0
5                    2AM              41             31                10
6                   2xLL              50             35                15
7               3 CAÍDAS              13             13                 0
8                 4 PREZ              35             35                 0
9                 4TO 23              17             17                 0


In [29]:
print(
    "Average rank improvement:",
    round(song_lifecycle["rank_improvement"].mean(), 2)
)

print(
    "Songs that improved after entry:",
    (song_lifecycle["rank_improvement"] > 0).sum()
)

print(
    "Songs that entered at peak:",
    (song_lifecycle["rank_improvement"] == 0).sum()
)

print(
    "Songs whose peak was worse than entry:",
    (song_lifecycle["rank_improvement"] < 0).sum()
)

Average rank improvement: 8.27
Songs that improved after entry: 283
Songs that entered at peak: 300
Songs whose peak was worse than entry: 0


In [30]:
latest_position = (
    df.sort_values("date")
      .groupby(["song", "artist"])
      .last()
      .reset_index()[["song", "artist", "position", "date"]]
      .rename(columns={
          "position": "latest_position",
          "date": "latest_date"
      })
)

song_lifecycle = song_lifecycle.merge(
    latest_position,
    on=["song", "artist"],
    how="left"
)

In [31]:
song_lifecycle["lifecycle_age_days"] = (
    song_lifecycle["latest_date"]
    - song_lifecycle["entry_date"]
).dt.days + 1

In [32]:
print(
    song_lifecycle[
        [
            "song",
            "entry_date",
            "peak_position",
            "entry_position",
            "latest_position",
            "days_on_playlist",
            "lifecycle_age_days"
        ]
    ].head(10)
)

                    song entry_date  peak_position  entry_position  \
0            0 CONFIANZA 2025-06-02             23              40   
1  1 de Enero, Puntacana 2025-04-28             45              45   
2              1000COSAS 2025-03-10              3               3   
3             100xCiento 2025-04-30             13              42   
4            14 FEBREROS 2025-11-24              9               9   
5                    2AM 2025-11-24             31              41   
6                   2xLL 2025-07-12             35              50   
7               3 CAÍDAS 2025-04-28             13              13   
8                 4 PREZ 2024-05-28             35              35   
9                 4TO 23 2025-07-07             17              17   

   latest_position  days_on_playlist  lifecycle_age_days  
0               42                22                  22  
1               45                 1                   1  
2               44               249                

In [33]:
def classify_lifecycle(row):
    if row["lifecycle_age_days"] <= 7:
        return "New Entry"
    
    elif row["latest_position"] <= 10:
        return "Peak Phase"
    
    elif row["latest_position"] <= 30:
        return "Mature Phase"
    
    elif row["latest_position"] > row["entry_position"]:
        return "Decline Phase"
    
    else:
        return "Growth Phase"


song_lifecycle["lifecycle_stage"] = (
    song_lifecycle.apply(classify_lifecycle, axis=1)
)

In [34]:
print(
    song_lifecycle["lifecycle_stage"]
    .value_counts()
)

lifecycle_stage
Decline Phase    248
New Entry        233
Growth Phase      69
Mature Phase      24
Peak Phase         9
Name: count, dtype: int64


In [35]:
stage_distribution = (
    song_lifecycle["lifecycle_stage"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(stage_distribution)

lifecycle_stage
Decline Phase    42.54
New Entry        39.97
Growth Phase     11.84
Mature Phase      4.12
Peak Phase        1.54
Name: proportion, dtype: float64


In [36]:
song_lifecycle["current_decline_from_peak"] = (
    song_lifecycle["latest_position"]
    - song_lifecycle["peak_position"]
)

In [37]:
print(
    song_lifecycle[
        [
            "song",
            "entry_position",
            "peak_position",
            "latest_position",
            "current_decline_from_peak"
        ]
    ].head(10)
)

                    song  entry_position  peak_position  latest_position  \
0            0 CONFIANZA              40             23               42   
1  1 de Enero, Puntacana              45             45               45   
2              1000COSAS               3              3               44   
3             100xCiento              42             13               37   
4            14 FEBREROS               9              9               16   
5                    2AM              41             31               31   
6                   2xLL              50             35               47   
7               3 CAÍDAS              13             13               47   
8                 4 PREZ              35             35               46   
9                 4TO 23              17             17               50   

   current_decline_from_peak  
0                         19  
1                          0  
2                         41  
3                         24  
4       

In [38]:
print(
    "Average movement from peak:",
    round(
        song_lifecycle["current_decline_from_peak"].mean(),
        2
    )
)

print(
    "Songs currently at their peak:",
    (
        song_lifecycle["current_decline_from_peak"] == 0
    ).sum()
)

print(
    "Songs currently below their peak:",
    (
        song_lifecycle["current_decline_from_peak"] > 0
    ).sum()
)

Average movement from peak: 19.73
Songs currently at their peak: 120
Songs currently below their peak: 463


In [39]:
def classify_lifecycle(row):
    age = row["lifecycle_age_days"]
    current = row["latest_position"]
    entry = row["entry_position"]
    peak = row["peak_position"]

    if age <= 7:
        return "New Entry"

    elif current <= 10:
        return "Peak Phase"

    elif current - peak >= 15:
        return "Decline Phase"

    elif current <= 30:
        if current <= entry:
            return "Mature Phase"
        else:
            return "Decline Phase"

    elif current <= entry:
        return "Growth Phase"

    else:
        return "Decline Phase"


song_lifecycle["lifecycle_stage"] = (
    song_lifecycle.apply(classify_lifecycle, axis=1)
)

In [40]:
stage_distribution = (
    song_lifecycle["lifecycle_stage"]
    .value_counts()
    .to_frame("count")
)

stage_distribution["percentage"] = (
    stage_distribution["count"]
    / len(song_lifecycle)
    * 100
).round(2)

print(stage_distribution)

                 count  percentage
lifecycle_stage                   
Decline Phase      299       51.29
New Entry          233       39.97
Growth Phase        31        5.32
Mature Phase        11        1.89
Peak Phase           9        1.54


In [41]:
song_lifecycle["retention_stability_index"] = (
    song_lifecycle["observed_playlist_days"]
    / song_lifecycle["days_on_playlist"]
    * 100
).round(2)

In [42]:
print(
    "Minimum stability:",
    song_lifecycle["retention_stability_index"].min()
)

print(
    "Maximum stability:",
    song_lifecycle["retention_stability_index"].max()
)

print(
    "Average stability:",
    round(
        song_lifecycle["retention_stability_index"].mean(),
        2
    )
)

print(
    "Missing stability values:",
    song_lifecycle["retention_stability_index"]
    .isnull()
    .sum()
)

Minimum stability: 3.68
Maximum stability: 100.0
Average stability: 91.75
Missing stability values: 0


In [43]:
song_attributes = (
    df.groupby(["song", "artist"])
      .agg(
          is_explicit=("is_explicit", "first"),
          album_type=("album_type", "first"),
          duration_ms=("duration_ms", "first"),
          total_tracks=("total_tracks", "first")
      )
      .reset_index()
)

song_attributes.head(10)

,song,artist,is_explicit,album_type,duration_ms,total_tracks
0,0 CONFIANZA,Rvfv,False,album,217500,20
1,"1 de Enero, Puntacana",Rels B,False,album,95000,13
2,1000COSAS,Lola Indigo & Manuel Turizo,False,single,239482,1
3,100xCiento,Foreign Teck & Eladio Carrion & De La Ghetto,True,single,219682,1
4,14 FEBREROS,Quevedo,True,album,172422,18
5,2AM,Sebastian Yatra & Bad Gyal,False,single,196426,1
6,2xLL,JC Reyes & YOVNGCHIMI & Slayter,False,single,178909,1
7,3 CAÍDAS,SAIKO,False,album,251565,16
8,4 PREZ,Cruz Cafuné,True,album,149484,23
9,4TO 23,Aitana,False,single,168320,1


In [44]:
song_lifecycle = song_lifecycle.merge(
    song_attributes,
    on=["song", "artist"],
    how="left"
)

In [45]:
print(
    song_lifecycle[
        [
            "song",
            "artist",
            "is_explicit",
            "album_type",
            "duration_ms",
            "total_tracks"
        ]
    ].head(10)
)

                    song                                        artist  \
0            0 CONFIANZA                                          Rvfv   
1  1 de Enero, Puntacana                                        Rels B   
2              1000COSAS                   Lola Indigo & Manuel Turizo   
3             100xCiento  Foreign Teck & Eladio Carrion & De La Ghetto   
4            14 FEBREROS                                       Quevedo   
5                    2AM                    Sebastian Yatra & Bad Gyal   
6                   2xLL               JC Reyes & YOVNGCHIMI & Slayter   
7               3 CAÍDAS                                         SAIKO   
8                 4 PREZ                                   Cruz Cafuné   
9                 4TO 23                                        Aitana   

   is_explicit album_type  duration_ms  total_tracks  
0        False      album       217500            20  
1        False      album        95000            13  
2        False     s

In [46]:
print("Missing explicit values:",
      song_lifecycle["is_explicit"].isnull().sum())

print("Missing album types:",
      song_lifecycle["album_type"].isnull().sum())

print("Missing duration values:",
      song_lifecycle["duration_ms"].isnull().sum())

print("Missing album track counts:",
      song_lifecycle["total_tracks"].isnull().sum())

Missing explicit values: 0
Missing album types: 0
Missing duration values: 0
Missing album track counts: 0


In [47]:
print("\nExplicit content:")
print(song_lifecycle["is_explicit"].value_counts())

print("\nAlbum type:")
print(song_lifecycle["album_type"].value_counts())


Explicit content:
is_explicit
False    329
True     254
Name: count, dtype: int64

Album type:
album_type
album     293
single    290
Name: count, dtype: int64


In [48]:
song_lifecycle["duration_seconds"] = (
    song_lifecycle["duration_ms"] / 1000
).round(2)

In [49]:
song_lifecycle["content_type"] = (
    song_lifecycle["is_explicit"]
    .map({
        True: "Explicit",
        False: "Clean"
    })
)

In [50]:
song_lifecycle["release_type"] = (
    song_lifecycle["album_type"]
    .str.title()
)

In [51]:
print(
    song_lifecycle[
        [
            "song",
            "duration_seconds",
            "content_type",
            "release_type"
        ]
    ].head(10)
)

                    song  duration_seconds content_type release_type
0            0 CONFIANZA            217.50        Clean        Album
1  1 de Enero, Puntacana             95.00        Clean        Album
2              1000COSAS            239.48        Clean       Single
3             100xCiento            219.68     Explicit       Single
4            14 FEBREROS            172.42     Explicit        Album
5                    2AM            196.43        Clean       Single
6                   2xLL            178.91        Clean       Single
7               3 CAÍDAS            251.56        Clean        Album
8                 4 PREZ            149.48     Explicit        Album
9                 4TO 23            168.32        Clean       Single


In [52]:
print("Content types:")
print(song_lifecycle["content_type"].value_counts())

print("\nRelease types:")
print(song_lifecycle["release_type"].value_counts())

Content types:
content_type
Clean       329
Explicit    254
Name: count, dtype: int64

Release types:
release_type
Album     293
Single    290
Name: count, dtype: int64


In [53]:
def duration_category(seconds):
    if seconds < 180:
        return "Short (<3 min)"
    elif seconds <= 240:
        return "Medium (3–4 min)"
    else:
        return "Long (>4 min)"

song_lifecycle["duration_category"] = (
    song_lifecycle["duration_seconds"]
    .apply(duration_category)
)

In [54]:
print(
    song_lifecycle["duration_category"]
    .value_counts()
)

duration_category
Medium (3–4 min)    273
Short (<3 min)      231
Long (>4 min)        79
Name: count, dtype: int64


In [55]:
entry_popularity = (
    df.sort_values("date")
      .groupby(["song", "artist"])
      .first()
      .reset_index()[["song", "artist", "popularity"]]
      .rename(columns={"popularity": "entry_popularity"})
)

song_lifecycle = song_lifecycle.merge(
    entry_popularity,
    on=["song", "artist"],
    how="left"
)

In [56]:
peak_popularity = (
    df.groupby(["song", "artist"])["popularity"]
      .max()
      .reset_index(name="peak_popularity")
)

song_lifecycle = song_lifecycle.merge(
    peak_popularity,
    on=["song", "artist"],
    how="left"
)

In [57]:
latest_popularity = (
    df.sort_values("date")
      .groupby(["song", "artist"])
      .last()
      .reset_index()[["song", "artist", "popularity"]]
      .rename(columns={"popularity": "latest_popularity"})
)

song_lifecycle = song_lifecycle.merge(
    latest_popularity,
    on=["song", "artist"],
    how="left"
)

In [58]:
song_lifecycle["popularity_growth"] = (
    song_lifecycle["peak_popularity"]
    - song_lifecycle["entry_popularity"]
)

In [59]:
print(
    song_lifecycle[
        [
            "song",
            "entry_popularity",
            "peak_popularity",
            "latest_popularity",
            "popularity_growth"
        ]
    ].head(10)
)

                    song  entry_popularity  peak_popularity  \
0            0 CONFIANZA                35               62   
1  1 de Enero, Puntacana                65               65   
2              1000COSAS                65               81   
3             100xCiento                72               75   
4            14 FEBREROS                63               68   
5                    2AM                55               64   
6                   2xLL                61               71   
7               3 CAÍDAS                61               69   
8                 4 PREZ                53               56   
9                 4TO 23                58               69   

   latest_popularity  popularity_growth  
0                 62                 27  
1                 65                  0  
2                 72                 16  
3                 75                  3  
4                 68                  5  
5                 64                  9  
6           

In [60]:
print("Missing entry popularity:",
      song_lifecycle["entry_popularity"].isnull().sum())

print("Missing peak popularity:",
      song_lifecycle["peak_popularity"].isnull().sum())

print("Missing latest popularity:",
      song_lifecycle["latest_popularity"].isnull().sum())

print(
    "Average popularity growth:",
    round(song_lifecycle["popularity_growth"].mean(), 2)
)

Missing entry popularity: 0
Missing peak popularity: 0
Missing latest popularity: 0
Average popularity growth: 18.04


In [61]:
explicit_lifecycle = (
    song_lifecycle
    .groupby("content_type")["observed_playlist_days"]
    .mean()
    .round(2)
)

print(explicit_lifecycle)

content_type
Clean       49.65
Explicit    44.94
Name: observed_playlist_days, dtype: float64


In [62]:
explicit_score = (
    explicit_lifecycle["Explicit"]
    / explicit_lifecycle["Clean"]
)

print(
    "Explicit Content Lifecycle Score:",
    round(explicit_score, 3)
)

Explicit Content Lifecycle Score: 0.905


In [63]:
explicit_comparison = (
    song_lifecycle
    .groupby("content_type")
    .agg(
        songs=("song", "count"),
        avg_observed_days=("observed_playlist_days", "mean"),
        avg_lifecycle_days=("days_on_playlist", "mean"),
        avg_peak_position=("peak_position", "mean"),
        avg_time_to_peak=("time_to_peak_days", "mean"),
        avg_stability=("retention_stability_index", "mean"),
        avg_popularity_growth=("popularity_growth", "mean")
    )
    .round(2)
)

print(explicit_comparison)

              songs  avg_observed_days  avg_lifecycle_days  avg_peak_position  \
content_type                                                                    
Clean           329              49.65               58.65              22.22   
Explicit        254              44.94               53.54              21.54   

              avg_time_to_peak  avg_stability  avg_popularity_growth  
content_type                                                          
Clean                     9.62          91.46                  14.73  
Explicit                  9.38          92.13                  22.32  


In [64]:
release_comparison = (
    song_lifecycle
    .groupby("release_type")
    .agg(
        songs=("song", "count"),
        avg_observed_days=("observed_playlist_days", "mean"),
        avg_lifecycle_days=("days_on_playlist", "mean"),
        avg_peak_position=("peak_position", "mean"),
        avg_time_to_peak=("time_to_peak_days", "mean"),
        avg_stability=("retention_stability_index", "mean"),
        avg_popularity_growth=("popularity_growth", "mean")
    )
    .round(2)
)

print(release_comparison)

              songs  avg_observed_days  avg_lifecycle_days  avg_peak_position  \
release_type                                                                    
Album           293              38.05               48.19              22.93   
Single          290              57.25               64.74              20.90   

              avg_time_to_peak  avg_stability  avg_popularity_growth  
release_type                                                          
Album                     8.61          92.03                  22.69  
Single                   10.42          91.47                  13.34  


In [65]:
single_album_ratio = (
    release_comparison.loc["Single", "avg_observed_days"]
    / release_comparison.loc["Album", "avg_observed_days"]
)

print(
    "Single vs Album Longevity Ratio:",
    round(single_album_ratio, 3)
)

Single vs Album Longevity Ratio: 1.505


In [66]:
song_lifecycle.to_csv(
    "../data/feature_engineered_data.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")
print("Rows:", len(song_lifecycle))
print("Columns:", len(song_lifecycle.columns))

Feature-engineered dataset saved successfully!
Rows: 583
Columns: 29


In [67]:
import os

file_path = "../data/feature_engineered_data.csv"

print("File exists:", os.path.exists(file_path))

if os.path.exists(file_path):
    print(
        "File size:",
        round(os.path.getsize(file_path) / 1024, 2),
        "KB"
    )

File exists: True
File size: 106.79 KB


In [68]:
import os

os.makedirs("../data", exist_ok=True)

df.to_csv("../data/feature_data.csv", index=False)

print("Feature data saved successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Feature data saved successfully!
Rows: 27750
Columns: 10


In [69]:
%whos

Variable              Type         Data/Info
--------------------------------------------
classify_lifecycle    function     <function classify_lifecy<...>le at 0x00000223EE930720>
df                    DataFrame    Shape: (27750, 10)
duration_category     function     <function duration_catego<...>ry at 0x00000223EE931220>
entry_popularity      DataFrame    Shape: (583, 3)
entry_position        DataFrame    Shape: (583, 3)
explicit_comparison   DataFrame    Shape: (2, 7)
explicit_lifecycle    Series       Shape: (2,)
explicit_score        float64      0.9051359516616314
file_path             str          ../data/feature_engineered_data.csv
latest_popularity     DataFrame    Shape: (583, 3)
latest_position       DataFrame    Shape: (583, 4)
np                    module       <module 'numpy' from 'c:\<...>ges\\numpy\\__init__.py'>
observed_days         DataFrame    Shape: (583, 3)
os                    module       <module 'os' (frozen)>
pd                    module       <module 'panda

In [70]:
import os

os.makedirs("../data", exist_ok=True)

song_lifecycle.to_csv(
    "../data/feature_data.csv",
    index=False
)

print("Feature data saved successfully!")
print("Rows:", len(song_lifecycle))
print("Columns:", len(song_lifecycle.columns))

Feature data saved successfully!
Rows: 583
Columns: 29


In [71]:
import os

print(
    "File exists:",
    os.path.exists("../data/feature_data.csv")
)

File exists: True
